#📚 文系大学生による『実践データ分析100本ノック＆LLM活用20本ノック』学習ログ
**著： 岡野 菜月（四国大学文学部 / 28卒）**

**GitHub： https://github.com/NatsukiOkano**

##📝 このノートブックの概要
＊ 書籍「Python 実践データ分析100本ノック 第3版」（秀和システム新社刊）の学習過程を記録したものです。

＊ 最大の特徴は、「文学部の視点でIT・データ分析を再解釈する」という試みにあります。

＊ 非情報系の学習者が直面する「専門用語の壁」を取り払うため、すべての処理に対し、論理的かつ直感的な「完全独自解説」を記述しています。

##⚠️ 注意事項
- **著作権保護の観点から、データセットおよび実行結果は同梱しておりません。**

### 教材
下山輝昌・松田雄馬・三木孝行 著「Python 実践データ分析100本ノック 第3版」（秀和システム新社、2025）

# 7章 ロジスティクスネットワークの最適設計を行う10本ノック

ここでは、最適化計算を行ういくつかのライブラリを用いて、最適化計算を実際に行っていきます。  
そして、前章で用いたネットワーク可視化などの技術を駆使し、計算結果の妥当性を確認する方法についても学んでいきます。

In [ ]:
#ロジスティクスネットワーク：物流ネットワーク。物の流れを中心とした経営全般的な問題。
#ネットワーク：点と線がつながり、物や情報が流れる仕組み。
#最適化：制約条件を満たして、目的関数を最小化あるいは最大化させること。
#制約条件：最小化または最大化を行うにあたって、守るべき条件。
#目的関数：最小化または最大化を目的として定義した関数。
#関数：計算式。
#アルゴリズム：実行手順。計算手順。処理手順。

### ノック６１：輸送最適化問題を解いてみよう

In [ ]:
#AIモデル：機械学習。予測する。大量のデータが必要。
#数理モデル：最適化。計算する。制約条件と目的関数が必要。

In [ ]:
!pip install pulp
!pip install ortoolpy

import numpy as np
import pandas as pd
from itertools import product
from pulp import LpVariable, lpSum, value
from ortoolpy import model_min, addvars, addvals

#!pip install：外部のライブラリをインストールして、端末内で実行待機させる。
# !：OSに直接実行命令をする。
#Pythonライブラリ自動導入（pip）：Pythonライブラリを端末に自動でインストールするためのシステム。
#install：インストールを開始して、自分の端末内で実行待機させる。
#※「!pip install」だけではプログラムで使えない。「import」する必要がある。

#ライブラリ（numpy）：「数値の塊」を操る。数学の天才。大量の数値を超高速で計算する。
#ライブラリ（pandas）：「データフレーム(表)の構造」を操る。例）並べ替え、抽出、結合、欠損値の穴埋め、CSVの読み書きなど
#ライブラリ（itertools）：データを結合して変数(LpVariable)に入れる素材を作成する。
#ライブラリ（pulp）：最適化に特化した数理モデルを作成する。
#ライブラリ（ortoolpy）：PuLPがデータを計算しやすい形に並べ替えたり、テンプレート(型)を作成したりする。
#関数（product）：データを結合して変数(LpVariable)に入れる素材を自動で作成する。
#クラス（LpVariable）：最適化を計算したい素材を入れる変数。
#関数（lpSum）：多くの変数(LpVariable)を足し合わせて目的関数や制約条件を作る。
#関数（value）：最適解(最適化の計算結果)を確認する。
#関数（model_min）：制約条件を満たして、目的関数を最小化させるための数理モデルを作成する。
#関数（addvars）：データ1行ごとの変数(LpVariable)を作成する。
#関数（addvals）：最適解(最適化の計算結果)をデータに反映させる。

In [ ]:
df_tc = pd.read_csv('trans_cost.csv', index_col='工場')
df_tc

#index_col：どの変数を「行ラベル兼index」として使うかを指定する。
#※実行すると、元のindex(行番号)は削除される。

In [ ]:
df_demand = pd.read_csv('demand.csv')
df_demand

In [ ]:
df_supply = pd.read_csv('supply.csv')
df_supply

In [ ]:
# 初期設定 #
np.random.seed(1)
nw = len(df_tc.index)
nf = len(df_tc.columns)
pr = list(product(range(nw), range(nf)))
pr

#np.random.rand()：「0以上1未満」の乱数を設定。
#np.random.seed(1)：乱数の発生パターン(型)を「1」が名前の設定に固定。
#乱数：規則性がなく、予測できない数字のこと。

#素材の自動作成（product）：データを結合して変数(LpVariable)に入れる素材を自動で作成する。
#list()：抽出したデータをPython標準の「リスト型」に変換する。
#range()：指定した回数分、実行を繰り返す。

In [ ]:
# 数理モデル(m1)と変数(v1)の作成 #
m1 = model_min()
v1 = {(i,j):LpVariable('dv%d_%d'%(i,j),lowBound=0) for i,j in pr}

#クラス（LpVariable）：最適化を計算したい素材を入れる変数。

#「'dv%d_%d'%(i,j)」=「'dvi_j'」
#「'dv%_%d'」= 文章, 「dv」= 文字, 「_」= 空白
#「%(i,j)」= 1つ目の「%d」に「i」を代入, 2つ目の「%d」に「j」を代入
#「%d」：整数(digit)を「%d」に代入。
#「%s」：文字(string)を「%s」に代入。

#lowBound：変数の下限値(最低値)を設定。
#lowBound=0：変数の下限値(最低値)を「0」に設定。

#「(i,j):」：「'dv%d_%d'%(i,j)」=「'dvi_j'」の変数名(検索用ラベル)。
#「for i,j in pr」：「pr(素材)」から「i(整数)」と「j(整数)」を抽出。

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[行番号,列番号]：整数で指定。整数を使って条件に合うデータを探す。

In [ ]:
# 目的関数と制約条件を設定して最適化計算を開始 #
m1 += lpSum(df_tc.iloc[i,j]*v1[i,j] for i,j in pr)
for i in range(nw):
    m1 += lpSum(v1[i,j] for j in range(nf)) <= df_supply.iloc[0,i]
for j in range(nf):
    m1 += lpSum(v1[i,j] for i in range(nw)) >= df_demand.iloc[0,j]
m1.solve()

# a += 1 ： a = a + 1
#関数（lpSum）：多くの変数を足し合わせて、目的関数や制約条件を作る。

#for i in range：「繰り返し処理」の基本セット
  # for：～の間、繰り返す。
  # in ：繰り返す回数。
  # round()：数値を四捨五入する。

#.solve()：最適化の計算を開始する。

In [ ]:
# 総輸送コスト計算 #
df_tr_sol = df_tc.copy()
total_cost = 0
for k,x in v1.items():
    i,j = k[0],k[1]
    df_tr_sol.iloc[i,j] = value(x)
    total_cost += df_tc.iloc[i,j] * value(x)

print(df_tr_sol)
print(f'総輸送コスト: {total_cost}')

#df.copy()：複製。コピーして新しいデータフレーム(表)を作る。

#辞書.items()：キー(文字列)とそれに対応する数値、それぞれに変数(k,x)を与える。
#辞書：変数 = {'文字列':数値, '文字列':数値, .....}。キー(文字列)とそれに対応する数値を結合させて、データを管理するデータ型。
#辞書（v1）：検索用ラベル(変数名)とそれに対応するLpVariableを結合させて、データを管理するデータ型。
#検索用ラベル（i,j = k[0],k[1]）：「i = k[0](検索用行番号[0])」,「j = k[1](検索用列番号[1])」

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[行番号,列番号]：整数で指定。整数を使って条件に合うデータを探す。
#※ iloc[i,j] = iloc[行番号,列番号]

#関数（value）：最適解(最適化の計算結果)を確認する。

### ノック６２：最適輸送ルートをネットワークで確認しよう

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

#ライブラリ（networkx）：ネットワーク構造を計算・操作・分析する。
#ライブラリ（matplotlib）：全ての工程を自らの手で制御する「プロ仕様版」グラフ作成ツール。
#ライブラリ（matplotlib.pyplot）：複雑な工程を自動化した「簡易操作版」グラフ作成ツール。

In [ ]:
df_tr = df_tr_sol.copy()
df_pos = pd.read_csv('trans_route_pos.csv')
print(df_tr.head())
print('---------------')
print(df_pos.head())

#df.copy()：複製。コピーして新しいデータフレーム(表)を作る。

In [ ]:
G = nx.Graph()

for i in range(len(df_pos.columns)):
    G.add_node(df_pos.columns[i])

num_pre = 0
edge_weights = []
size = 0.1
for i in range(len(df_pos.columns)):
    for j in range(len(df_pos.columns)):
        if not (i==j):
           G.add_edge(df_pos.columns[i], df_pos.columns[j])
           if num_pre<len(G.edges):
              num_pre = len(G.edges)
              weight = 0
              if (df_pos.columns[i] in df_tr.columns)and(df_pos.columns[j] in df_tr.index):
                  if df_tr[df_pos.columns[i]][df_pos.columns[j]]:
                     weight = df_tr[df_pos.columns[i]][df_pos.columns[j]] * size
              elif (df_pos.columns[j] in df_tr.columns)and(df_pos.columns[i] in df_tr.index):
                  if df_tr[df_pos.columns[j]][df_pos.columns[i]]:
                     weight = df_tr[df_pos.columns[j]][df_pos.columns[i]] * size
              edge_weights.append(weight)

pos = {}
for i in range(len(df_pos.columns)):
    node = df_pos.columns[i]
    pos[node] = (df_pos[node][0], df_pos[node][1])

nx.draw(G, pos, with_labels=True, font_size=16, node_size = 1000, node_color='k', font_color='w', width=edge_weights)

plt.show()

#nx.Graph()：ネットワーク図のデータ(オブジェクト)を作成する。

#node：点。頂点。
#add_node()：作成したネットワーク図に「新しい頂点(点)」を追加する。
#add_edge()：作成したネットワーク図に「新しい辺(edge)」を追加する。
#

#num_pre：変数。※「num = number(数), pre = previous(前の)」新しく辺(edge)を追加する前の辺(edge)の総数。
#num_pre = 0：辺(edge)が新しく追加されたか確認する。
#edge_weights：変数。※ edge = 辺(線) , weight = 辺(edge)の太さを決める数値。
#edge_weights = []：変数(edge_weights)にデータを入れるためのリスト([])を作成する。
#size：変数。※ size = 大きさ。

#append()：指定したリストごと最後に追加する。
#extend()：指定したリストのデータのみ最後に追加する。
#【例：[1, 2, 3]に、[4, 5]を追加。↓】
# append() [1, 2, 3, [4, 5]]、extend() [1, 2, 3, 4, 5]

#pos={}：ネットワーク図の頂点(点)の位置データを作成する。
#len：データの件数を数える。Excelのcount関数と同じ役割。

#nx.draw()：設定したデータ(オブジェクト)をネットワーク図に反映させる。
#with_labels=True：ネットワーク図に頂点(点)のラベル(名前)を表示する。
#with_labels=False：ネットワーク図に頂点(点)のラベル(名前)を表示しない。
#font_size：文字の大きさを設定。
#node_size：頂点(点)の大きさを設定。
#node_color：頂点(点)の色を設定。※ K = black(黒)
#font_color：頂点(点)ラベルの色を設定。※ w = white(白)
#width：線の太さを設定。

#plt.show()：作成したネットワーク図を表示(可視化)する。

### ノック６３：最適輸送ルートが制約条件内に収まっているかどうかを確認しよう

In [ ]:
df_demand = pd.read_csv('demand.csv')
df_supply = pd.read_csv('supply.csv')
print(df_demand.head())
print('---------------')
print(df_supply.head())

In [ ]:
# 制約条件計算関数

# 需要側
def condition_demand(df_tr, df_demand):
    flag = np.zeros(len(df_demand.columns))
    for i in range(len(df_demand.columns)):
        temp_sum = sum(df_tr[df_demand.columns[i]])
        if (temp_sum >= df_demand.iloc[0,i]):
            flag[i] = 1
    return flag

# 供給側
def condition_supply(df_tr, df_supply):
    flag = np.zeros(len(df_supply.columns))
    for i in range(len(df_supply.columns)):
        temp_sum = sum(df_tr.loc[df_supply.columns[i]])
        if temp_sum <= df_supply.iloc[0,i]:
           flag[i] = 1
    return flag

print(f'需要条件計算結果: {condition_demand(df_tr_sol, df_demand)}')
print(f'供給条件計算結果: {condition_supply(df_tr_sol, df_supply)}')

#def：「define(定義する)」の略。手順を関数として定義する。
#def condition_demand(df_tr, df_demand)：def 手順に名付けた関数名(変数1, 変数2)
#※ 関数 = 計算式。変数1, 変数2 = 定義する関数の素材。

#np.zeros()：要素(データ)が全て「0」の表またはリストを新しく作成する。

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[行番号,列番号]：整数で指定。整数を使って条件に合うデータを探す。
#※ iloc[i,j] = iloc[行番号,列番号]

In [ ]:
#線形最適化：限られた資源で「目的関数(利益や費用など)を最大化または最小化」する最適解を見つけ出す計算方法。
#線形最適化 = 線形計画法

### ノック６４：生産計画に関するデータを読み込んでみよう

In [ ]:
df_material = pd.read_csv('product_plan_material.csv', index_col='製品')
df_profit = pd.read_csv('product_plan_profit.csv', index_col='製品')
df_stock = pd.read_csv('product_plan_stock.csv', index_col='項目')
df_plan = pd.read_csv('product_plan.csv', index_col='製品')

print(df_material.head())
print('---------------')
print(df_profit.head())
print('---------------')
print(df_stock.head())
print('---------------')
print(df_plan.head())

### ノック６５：利益を計算する関数を作ってみよう

In [ ]:
# 利益計算関数
def product_plan(df_profit, df_plan):
    profit = 0
    for i in range(len(df_profit.index)):
        for j in range(len(df_plan.columns)):
            profit += df_profit.iloc[i,j] * df_plan.iloc[i,j]
    return profit

print(f'総利益: {product_plan(df_profit, df_plan)}')

#def：「define(定義する)」の略。定式化(計算の手順)を関数として定義する。
#def product_plan(df_profit, df_plan)：def 手順に名付けた関数名(変数1, 変数2)
#※ 関数 = 計算式。変数1, 変数2 = 定義する関数の素材。

# a += 1 ： a = a + 1

In [ ]:
#【最適化問題を解く方法 (2step↓)】
#①目的関数と制約条件を定義する。
#②制約条件を満たして、目的関数を最小化あるいは最大化させる変数の組み合わせを探す。

### ノック６６：生産最適化問題を解いてみよう

In [ ]:
from pulp import LpVariable, lpSum, value
from ortoolpy import model_max, addvars, addvals

#ライブラリ（pulp）：最適化に特化した数理モデルを作成する。
#ライブラリ（ortoolpy）：PuLPがデータを計算しやすい形に並べ替えたり、テンプレート(型)を作成したりする。

#クラス（LpVariable）：最適化を計算したい素材を入れる変数。
#関数（lpSum）：多くの変数を足し合わせて目的関数や制約条件を作る。
#関数（value）：最適解(最適化の計算結果)を確認する。

#関数（model_max）：制約条件を満たして、目的関数を最大化させるための数理モデルを作成する。
#関数（addvars）：データ1行ごとの変数(LpVariable)を作成する。
#関数（addvals）：最適解(最適化の計算結果)をデータに反映させる。

In [ ]:
df = df_material.copy()
inv = df_stock

m = model_max()
v1 = {(i):LpVariable('v%d'%(i), lowBound=0) for i in range(len(df_profit))}
m += lpSum(df_profit.iloc[i] * v1[i] for i in range(len(df_profit)))
for i in range(len(df_material.columns)):
    m += lpSum(df_material.iloc[j,i] * v1[j] for j in range(len(df_profit))) <= df_stock.iloc[:,i]
m.solve()

#df.copy()：複製。コピーして新しいデータフレーム(表)を作る。

#クラス（LpVariable）：最適化を計算したい素材を入れる変数。
#関数（lpSum）：多くの変数(LpVariable)を足し合わせて、目的関数や制約条件を作る。

#「'v%d'%(i)」=「'dvi'」
#「'v%d'」= 文章, 「v」= 文字
#「%(i)」= 「%d」に「i」を代入
#「%d」：整数(digit)を「%d」に代入。
#「%s」：文字(string)を「%s」に代入。
#lowBound：変数の下限値(最低値)を設定。
#lowBound=0：変数の下限値(最低値)を「0」に設定。
#「(i):」：「'v%d'%(i)」=「'dvi'」の変数名(検索用ラベル)。

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[行番号,列番号]：整数で指定。整数を使って条件に合うデータを探す。

#for i in range：「繰り返し処理」の基本セット
  # for：～の間、繰り返す。
  # i：今、何回目か数えるための変数。
  # in range()：繰り返す回数。

#.solve()：最適化の計算を開始する。

In [ ]:
df_plan_sol = df_plan.copy()
for k,x in v1.items():
    df_plan_sol.iloc[k] = value(x)

print(df_plan_sol)
print(f'総利益: {value(m.objective)}')

#クラス（LpVariable）：最適化を計算したい素材を入れる変数。
#関数（lpSum）：多くの変数を足し合わせて目的関数や制約条件を作る。
#関数（value）：最適解(最適化の計算結果)を確認する。

#辞書.items()：キー(文字列)とそれに対応する数値、それぞれに変数(k,x)を与える。
#辞書：変数 = {'文字列':数値, '文字列':数値, .....}。キー(文字列)とそれに対応する数値を結合させて、データを管理するデータ型。
#辞書（v1）：検索用ラベル(変数名)とそれに対応するLpVariableを結合させて、データを管理するデータ型。
#検索用ラベル（i,j = k[0],k[1]）：「i = k[0](検索用行番号[0])」,「j = k[1](検索用列番号[1])」

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[行番号,列番号]：整数で指定。整数を使って条件に合うデータを探す。
#※ iloc[i,j] = iloc[行番号,列番号]

### ノック６７：最適生産計画が制約条件内に収まっているかどうかを確認しよう

In [ ]:
# 制約条件計算関数
def condition_stock(df_plan, df_material,df_stock):
    flag = np.zeros(len(df_material.columns))
    for i in range(len(df_material.columns)):
        temp_sum = 0
        for j in range(len(df_material.index)):
            temp_sum = temp_sum + df_material.iloc[j,i] * float(df_plan.iloc[j,0])
        if (temp_sum <= float(df_stock.iloc[0,i])):
                flag[i] = 1
        print(f'{df_material.columns[i]} 使用量: {temp_sum}, 在庫: {float(df_stock.iloc[0,i])}')
    return flag

print(f'制約条件計算結果: {condition_stock(df_plan_sol, df_material, df_stock)}')

#def：「define(定義する)」の略。定式化(計算の手順)を関数として定義する。
#def condition_stock(df_plan, df_material,df_stock)：：def 手順に名付けた関数名(変数1, 変数2)
#※ 関数 = 計算式。変数1, 変数2 = 定義する関数の素材。

#np.zeros()：要素(データ)が全て「0」の表またはリストを新しく作成する。

### ノック６８：ロジスティクスネットワーク設計問題を解いてみよう

In [ ]:
製品 = list('AB')
需要地 = list('PQ')
工場 = list('XY')
レーン = (2,2)

#list()：抽出したデータをPython標準の「リスト型」に変換する。
#レーン = (2,2)：レーン = (x座標(行),y座標(列))

In [ ]:
# 輸送費表 #
tbdi = pd.DataFrame(((j,k) for j in 需要地 for k in 工場), columns=['需要地','工場'])
tbdi['輸送費'] = [1,2,3,1]
tbdi

#pd.DataFrame(データ,行ラベル,列ラベル)：データフレーム(表)を作成する。
#「(j,k) for j in 需要地 for k in 工場」：「j」に「需要地」を代入,「k」に「工場」を代入

In [ ]:
# 需用費 #
tbde = pd.DataFrame(((j,i) for j in 需要地 for i in 製品), columns=['需要地','製品'])
tbde['需要'] = [10,10,20,20]
tbde

In [ ]:
# 生産表 #
tbfa = pd.DataFrame(((k,l,i,0,np.inf) for k,nl in zip (工場,レーン) for l in range(nl) for i in 製品),columns=['工場','レーン','製品','下限','上限'])
tbfa['生産費'] = [1,np.nan,np.nan,1,3,np.nan,5,3]
tbfa.dropna(inplace=True)
tbfa.loc[4,'上限'] = 10
tbfa

#「(k,l,i,0,np.inf)」：変数。columns=['工場','レーン','製品','下限','上限']
#「k」工場,「l」レーン,「n」レーンの数※工場(k)が持つレーン(l)の数が違う可能性があるため「n」をつける。
#「zip(変数1,変数2)」：変数1と変数2の同じindex(行番号)でデータを結びつける。
#「np.nan」= 欠損値

#df.dropna()：全列が対象。1つでも欠損値(NaN)セルがあればその行ごと削除する。
#df.dropna(subset=['列名'])：指定した列だけが対象。欠損値(NaN)セルがあればその行ごと削除する。
#inplace：「『元のデータ』そのものに修正を加える」か、「『コピーした新しいデータ』に修正を加えてユーザーに渡す」か決める。
#inplace = True：『元のデータ』そのものに修正を加える。※ユーザーに渡さない。
#inplace = False：『コピーした新しいデータ』に修正を加えてユーザーに渡す。

#「inf」= 上限なし(無限大)

In [ ]:
from ortoolpy import logistics_network
_, tbdi2, _ = logistics_network(tbde,tbdi,tbfa)
tbfa

#ライブラリ（ortoolpy）：PuLPがデータを計算しやすい形に並べ替えたり、テンプレート(型)を作成したりする。

#関数（logistics_network）：物流ネットワークに特化した最適化問題を解く。
#関数の入力：logistics_network(df1,df2,df3) ※dfを3つ設定する。
#変数の出力：_, tbdi2, _ = 削除(tbde2), 表示(tbdi2), 削除(tbfa)
#※関数（logistics_network）は3つの表(計算結果)を表示するため、不要な表は「_」で削除する。

In [ ]:
tbdi2

#【関数（logistics_network）の戻り値↓】
#VarX：最適解の管理番号
#ValX：最適解(最適化の計算結果)
#戻り値(return)：実行結果

### ノック６９：最適ネットワークにおける輸送コストとその内訳を計算しよう

In [ ]:
print(tbdi2)
trans_cost = 0
for i in range(len(tbdi2.index)):
    trans_cost += tbdi2['輸送費'].iloc[i] * tbdi2['ValX'].iloc[i]
print(f'総輸送コスト: {trans_cost}')

#df.loc[行ラベル,列ラベル]：ラベルで指定。ラベルを使って条件に合うデータを探す。
#df.iloc[行番号,列番号]：整数で指定。整数を使って条件に合うデータを探す。
#※ iloc[i,j] = iloc[行番号,列番号]

# a += 1 ： a = a + 1

### ノック７０：最適ネットワークにおける生産コストとその内訳を計算しよう

In [ ]:
print(tbfa)
product_cost = 0
for i in range(len(tbfa.index)):
    product_cost += tbfa['生産費'].iloc[i] * tbfa['ValY'].iloc[i]
print(f'総生産コスト:{product_cost}')